# Are inversions in repeat regions?
Now that we did some repeat modeling and classification, we can see if the breakpoints of any of the simulated inversions fall into these identified repeat regions.

In [28]:
library(dplyr)
library(tidyr)

In [29]:
repeats <- read.table("repeat.regions", header = T)
repeats <- repeats[order(repeats$contig), ]
head(repeats)

,contig,position_start,position_end,class
,<chr>,<int>,<int>,<chr>
1,2L,11713984,11714447,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
2,2L,11714507,11715213,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
3,2L,11715209,11720435,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
4,2L,11720465,11720573,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
5,2L,1220606,1221069,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
6,2L,1221129,1221835,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy


We'll to read in the inventory of all inversion positions and pull out the unique inversions. I really should have committed to either comma or tab separated tables.

In [30]:
inversions <- read.csv("../assess_called_sv/linkedread.sv.assessment", header = T)
inversions <- unique(inversions[, c(1,6,10,11)])
rownames(inversions) <- NULL
inversions$is_repeat <- as.character(NA)
head(inversions)
nrow(inversions)

,contig,size,position_start,position_end,is_repeat
,<chr>,<chr>,<int>,<int>,<chr>
1,2L,small,3193196,3214221,NA
2,2L,small,3940211,3944863,NA
3,2L,small,13024404,13026577,NA
4,2L,small,14845709,14861200,NA
5,2L,small,20107901,20113695,NA
6,2R,small,4328632,4333509,NA


[1] 48

To do the matchy-matchy, we can borrow the fuzzy matching function we used to assess inversion calling. It will need some modifications to accommodate the slightly different use case.

In [31]:
for(i in 1:nrow(inversions)){
    .row <- inversions[i,]
    query <- which(
        repeats$contig == .row$contig &
        (repeats$position_start - 150 <= .row$position_start & repeats$position_end + 150 >= .row$position_start) |
        (repeats$position_start - 150 <= .row$position_end) & (repeats$position_end + 150 >= .row$position_end)
    )
    if(length(query) > 0) {
        inversions$is_repeat[i] <- do.call("paste", as.list(repeats$class[query]))
    }
}

In [32]:
are_repeats <- inversions[!is.na(inversions$is_repeat),]
are_repeats

,contig,size,position_start,position_end,is_repeat
,<chr>,<chr>,<int>,<int>,<chr>
1,2L,small,3193196,3214221,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;LINE;Group-II;Group-2;R1-like;CR1-group;CR1
2,2L,small,3940211,3944863,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
5,2L,small,20107901,20113695,Interspersed_Repeat;Transposable_Element;Class_II_DNA_Transposition;Transposase;CACTA;Transib Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Ty1-Copia
6,2R,small,4328632,4333509,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;LINE;Group-II;Group-2;R1-like;R1-group;I-group;Jockey Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
16,3R,small,20006864,20010483,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
21,2L,medium,1591146,1776073,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy Interspersed_Repeat;Transposable_Element;Class_II_DNA_Transposition;Transposase;CACTA;Transib Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
25,2R,medium,9640874,9978281,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy
33,3R,medium,16760480,16905871,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Bel-Pao
34,3R,medium,22722816,22831351,Interspersed_Repeat;Transposable_Element;Class_I_Retrotransposition;Retrotransposon;Long_Terminal_Repeat_Element;Gypsy-ERV;Gypsy


Let's find which inversions had the most false negatives.

In [33]:
called_sv <- read.csv("../assess_called_sv/linkedread.sv.assessment", header = T)
false_neg <- filter(called_sv, assessment == "false negative") %>%
    group_by(contig, position_start, size) %>%
    summarize(n = length(depth)) %>%
    arrange(desc(n))
false_neg$is_repeat <- FALSE
head(false_neg, 20)

`summarise()` has grouped output by 'contig', 'position_start'. You can
override using the `.groups` argument.


contig,position_start,size,n,is_repeat
<chr>,<int>,<chr>,<int>,<lgl>
2L,1591146,medium,55,FALSE
2L,6436329,medium,55,FALSE
2R,4328632,small,55,FALSE
3R,16760480,medium,40,FALSE
3R,26216293,medium,30,FALSE
2R,9640874,medium,26,FALSE
2R,17910285,medium,26,FALSE
2L,961650,xl,25,FALSE
2R,15862083,medium,25,FALSE


We can now combine the inversions-in-repeats with the inversions-not-detected dataframe to see if there's a correlation of hard-to-call inversions being associated with repeat regions.

In [36]:
for(i in 1:nrow(false_neg)){
    .row <- false_neg[i,]
    query <- which(
        are_repeats$contig == .row$contig &
        are_repeats$position_start == .row$position_start
    )
    if(length(query) > 0) {
        false_neg$is_repeat[i] <- TRUE
    }
}
arrange(false_neg, desc(is_repeat))

contig,position_start,size,n,is_repeat
<chr>,<int>,<chr>,<int>,<lgl>
2L,1591146,medium,55,TRUE
2R,4328632,small,55,TRUE
3R,16760480,medium,40,TRUE
3R,26216293,medium,30,TRUE
2R,9640874,medium,26,TRUE
2L,20107901,small,20,TRUE
2L,3940211,small,19,TRUE
2L,18872129,large,17,TRUE
3R,24237377,medium,17,TRUE
